In [6]:
import sys
import os

# Add embed folder to path
sys.path.insert(0, os.path.abspath("embed"))

# Download model files
os.chdir("embed")
import download
download.download("Xenova/all-MiniLM-L6-v2")
os.chdir("..")

# Load embedder
from embedder import Embedder
embedder = Embedder(path="embed/models/Xenova/all-MiniLM-L6-v2")
print("Embedder ready.")

  exists models/Xenova/all-MiniLM-L6-v2/tokenizer.json
  exists models/Xenova/all-MiniLM-L6-v2/model.onnx
Embedder ready.


In [7]:

from sentence_transformers import SentenceTransformer
import ingest
import os
os.environ["OPENAI_API_KEY"] = "your-key-here" 

In [49]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [50]:
from ingest import load_faq_data
documents = load_faq_data()
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm

from pydantic import BaseModel
class Questions(BaseModel):
    questions: list[str]

In [51]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [52]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()



In [53]:
import json
doc = documents[0]
user_prompt = json.dumps(doc)

In [54]:
from evaluation_utils import llm_structured

In [55]:
total_input = 0

for doc in documents[:3]:
    user_prompt = json.dumps(doc)
    result, usage = llm_structured(openai_client, data_gen_instructions, user_prompt, Questions)
    total_input += usage.input_tokens
    print(f"input tokens: {usage.input_tokens}")

print(f"Average input tokens: {total_input / 3:.0f}")

input tokens: 242
input tokens: 273
input tokens: 350
Average input tokens: 288


In [56]:
###The full ground truth

In [57]:
###Load it with pandas into a list of records called ground_truth. 
###Each record has a question and the filename of the page that should answer it.
import pandas as pd

df_ground_truth = pd.read_csv("/Users/arnenyecknyeck/Desktop/llmzoomcamp26/04-Evaluation/ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [58]:
### Searching the chunks
###We search over the same chunks as in homework 2.Create them with chunk_documents:

In [ ]:
from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
print(f"Documents: {len(documents)}")
print(documents[0].keys())

chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Chunks: {len(chunks)}")
print(chunks[0].keys())

In [ ]:
#Index the same chunks with Index from minsearch. Use content as a text field.(Text search)
from minsearch import Index

text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)


In [ ]:
# Vector search
texts = [chunk["content"] for chunk in chunks]
X = embedder.encode_batch(texts)

from minsearch import VectorSearch
vector_index = VectorSearch(keyword_fields=["filename"])
vector_index.fit(X, chunks)



def vector_search(query, num_results=5):
    q_vec = embedder.encode(query)
    return vector_index.search(q_vec, num_results=num_results)

In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [ ]:
## hybrid search
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [ ]:
q = ground_truth[0]["question"]
q

In [ ]:
text_results = text_search(q, num_results=10)
vector_results = vector_search(q, num_results=10)

In [ ]:
results = rrf([text_results, vector_results])
# results
print(results[0]['filename'])

In [ ]:
#Q3. First result with vector search

In [ ]:
# Q4 Evaluation 
import pandas as pd

df_ground_truth = pd.read_csv("/Users/arnenyecknyeck/Desktop/llmzoomcamp26/04-Evaluation/ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [ ]:
def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)

In [ ]:
q = ground_truth[0]
q

In [ ]:
for d in text_search(q["question"]):
    print(f'{d["filename"]} == {q["filename"]}: {d["filename"] == q["filename"]}')

In [ ]:
relevance = []
for d in results:
    relevance.append(int(d["filename"] == q["filename"]))
relevance

In [ ]:
def compute_relevance(q, search_function):
    filename = q["filename"]
    results = search_function(query=q["question"])
    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == filename))
    return relevance

In [ ]:
from tqdm.auto import tqdm

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []
    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)
    return relevance_total

In [ ]:
ground_truth_sample = ground_truth[:15]
relevance_total = compute_relevance_total(ground_truth, text_search)

In [ ]:
relevance_total

In [ ]:
cnt = 0

for line in relevance_total:
    if 1 in line:
        cnt = cnt + 1

cnt

In [ ]:
cnt / len(relevance_total)

In [ ]:
## EVALUATING VECTOR SEARCH

In [ ]:
relevance_total_vector = compute_relevance_total(ground_truth, vector_search)
relevance_total_vector

In [ ]:
cnt2 = 0
for line in relevance_total_vector:
    if 1 in line:
        cnt2 += 1

In [ ]:
cnt2 / len(relevance_total_vector)

In [ ]:
total_score = 0.0
for line in relevance_total_vector:
    for rank in range(len(line)):
        if line[rank] == 1:
            total_score += 1 / (rank + 1)
            break

mrr_score = total_score / len(relevance_total_vector)
print(f"Vector MRR: {mrr_score:.4f}")

In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

for k in [1, 50, 100, 200]:
    def hybrid_search(query, num_results=5):
        text_results = text_search(query, num_results=10)
        vector_results = vector_search(query, num_results=10)
        return rrf([text_results, vector_results], k=k)
    
    relevance_total_hybrid = compute_relevance_total(ground_truth, hybrid_search)
    
    total_score = 0.0
    for line in relevance_total_hybrid:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score += 1 / (rank + 1)
                break
    mrr_score = total_score / len(relevance_total_hybrid)
    print(f"k={k}: MRR={mrr_score:.4f}")